# Testing optic-flow vs. vectorial encoding in marching locusts

This notebook revisits and re-analyzes one set of the virtual-reality experiments of S. Sayin et al., "The behavioral mechanisms governing collective motion in swarming locusts," *Science* **387**(6737), 995-1000 (2025); [doi:10.1126/science.adq7832](https://doi.org/10.1126/science.adq7832): it fits a **hierarchical mixture-of-two-von Mises** model to the heading decisions of **52 juvenile desert locusts** (2435 observed "steps" total) that marched on a motion-compensating ball inside a panoramic virtual-reality arena, viewing moving **virtual conspecifics** placed symmetrically at a lateral distance of 8 or 50 cm. Each row of the step table is one discretized heading decision (radians, $-\pi$ to $\pi$) of one animal, with $0$ pointing forward along the global march direction; the raw per-animal recordings come from [`jgraving/unwrap-data`](https://github.com/jgraving/unwrap-data) and the first data cell runs the rediscretization pipeline that produces this table.

**Why a mixture?**  In the VR task, two moving virtual swarms were positioned symmetrically at the animal's flanks, and the behavioral readout is the *full distribution* of headings. Two mechanistic hypotheses were proposed for how this local motion drives directional choice:

- **optic–flow / optomotor** — the animal aligns with the wide-field motion and keeps
  marching forward (heading near $0$);
- **vectorial / ring-attractor** — the animal treats the flanks as spatial targets and
  bifurcates left or right to join them (headings near $\pm\tfrac{\pi}{2}$).

A single von Mises mode cannot express both attractors at once. The bimodal mixture gives
each animal two preferred directions with their own concentrations and a mixing weight
(a sigmoid rather than a softmax link), and the hierarchical layer partially pools
parameters across the 52 animals so that sparsely sampled animals (3 to 204 steps per
animal) are shrunk toward the population mixture.

**Structure of the notebook**

1. **Data loading & exploration** — pull the raw trajectories from `unwrap-data`,
   rediscretize them into heading steps, and count steps per animal.
2. **Bimodal model fit** — the full hierarchical mixture model (documented in the
   markdown cell below this section).
3. **Unimodal baseline** — a single-mode von Mises per animal with the same hierarchical
   structure, so the comparison is a controlled test of the second mode.
4. **Model comparison** — PSIS-LOO ELPD of the two fits, plus a population-level
   posterior predictive check against bootstrap-histogram and von Mises KDE baselines.
5. **Individual-level checks** — per-animal predictive density, mixture-component arrows,
   and lack-of-fit residuals.
6. **Bayesian hypothesis tests** — the population mixture's posterior density at $0$
   (optic flow) vs. $\pm\tfrac{\pi}{2}$ (vectorial encoding), against the uniform
   random-walk baseline, plus per-animal versions.

This R port reads the same raw recordings and reproduces the preprocessing,
priors, likelihoods, estimands, and figure basenames. It uses brms with
custom Stan likelihoods and explicit circular coefficient-vector priors.


## Setup

Select the **R (`ir`)** kernel after following the [R setup](../getting-started-r.md)
page. Run the cells in order from `notebooks_r/` or the repository root.
Notebook execution is local by design (tests/test_notebooks.py).
This draft has no retained outputs; installation, sampling, diagnostics, and
rendering are for the local execution pass.


In [ ]:
required <- c("brms", "posterior", "loo", "arrow", "circular",
              "ggplot2", "patchwork", "ragg", "IRdisplay")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) install.packages(missing, repos = "https://cloud.r-project.org")
if (!requireNamespace("cmdstanr", quietly = TRUE)) {
  install.packages("cmdstanr",
    repos = c("https://stan-dev.r-universe.dev", "https://cloud.r-project.org"))
}
cmdstan_ok <- tryCatch(dir.exists(as.character(cmdstanr::cmdstan_path())),
  error = function(e) FALSE)
if (!cmdstan_ok) cmdstanr::install_cmdstan()


In [ ]:
versions <- c(R = as.character(getRversion()),
  vapply(c(required, "cmdstanr"), function(p) as.character(packageVersion(p)), character(1)))
versions


In [ ]:
suppressPackageStartupMessages({
  library(brms)
  library(posterior)
  library(loo)
  library(circular)
  library(ggplot2)
  library(patchwork)
})


In [ ]:
repo_root <- if (file.exists(file.path("..", "R", "unwrap_functions.R"))) {
  ".."
} else if (file.exists(file.path("R", "unwrap_functions.R"))) {
  "."
} else {
  stop("Run from notebooks_r/ or the repository root.")
}
source(file.path(repo_root, "R", "unwrap_functions.R"))


In [ ]:
FIG_DIR <- file.path(repo_root, "figures", "locust_direction_mixture_model")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)


In [ ]:
show_figure <- function(plot, name, width, height, res = 110) {
  file <- file.path(FIG_DIR, paste0(name, ".png"))
  ragg::agg_png(file, width = width, height = height, units = "in",
                res = res)
  print(plot)
  invisible(dev.off())
  IRdisplay::display_png(file = file)
  invisible(NULL)
}


In [ ]:
theme_set(theme_minimal(base_size = 12))
save_figure <- function(plot, name, width = 8, height = 5) {
  ggsave(file.path(FIG_DIR, paste0(name, ".pdf")), plot,
         width = width, height = height)
  show_figure(plot, name, width, height)
}
SEED <- 42L
quick_run <- Sys.getenv("UNWRAP_R_QUICK") == "1"
CHAINS <- if (quick_run) 2L else 4L
ITER <- if (quick_run) 4000L else 8000L
WARMUP <- if (quick_run) 2000L else 4000L
N_BOOTSTRAPS <- if (quick_run) 150L else 1000L
N_BOOT_KDE <- if (quick_run) 100L else 1000L
# These are execution budgets, not convergence claims. Keep the complete
# dataset and both model structures; hard gates apply at either budget.
# The Python unimodal reference reports some R-hat/ESS breaches, so this
# local draft gives both fits more draws without relaxing the thresholds.


## Figure style

All figures render inline as headless ragg PNGs and save vector PDFs under
`figures/locust_direction_mixture_model/`. Basenames match the Python twin,
which uses `figures/locust/`. Colors retain orange for bimodal/vectorial,
blue for unimodal/optic flow, and red for predictive model densities.
Typography uses portable sans text. Line plots use monotone iteration or
angle grids; XY tracks use explicit successive-point segments.


In [ ]:
wrap <- function(x) (x + pi) %% (2 * pi) - pi
circ_mean <- function(x) atan2(mean(sin(x)), mean(cos(x)))
resultant <- function(x) Mod(mean(exp(1i * x)))
softplus <- function(x) pmax(x, 0) + log1p(exp(-abs(x)))
inverse_softplus <- function(x) log(expm1(x))
vm_logpdf <- function(y, mu, kappa) {
  kappa * (cos(y - mu) - 1) - log(2 * pi) -
    log(besselI(kappa, 0, expon.scaled = TRUE))
}
log_add <- function(a, b) {
  m <- pmax(a, b)
  m + log(exp(a - m) + exp(b - m))
}
A1_safe <- function(kappa) {
  besselI(kappa, 1, expon.scaled = TRUE) / besselI(kappa, 0, expon.scaled = TRUE)
}
hdi <- function(x, prob = 0.95) {
  x <- sort(as.numeric(x))
  stopifnot(length(x) > 1L, all(is.finite(x)))
  gap <- min(length(x) - 1L, floor(prob * length(x)))
  starts <- seq_len(length(x) - gap)
  j <- starts[which.min(x[starts + gap] - x[starts])]
  c(low = x[j], high = x[j + gap])
}
# Every prediction/density matrix in this notebook is [grid x draws].
vm_grid <- function(theta, mu, kappa) {
  stopifnot(length(mu) == length(kappa))
  G <- length(theta)
  K <- matrix(rep(kappa, each = G), nrow = G)
  delta <- outer(theta, mu, "-")
  exp(K * (cos(delta) - 1) - log(2 * pi) -
    matrix(rep(log(besselI(kappa, 0, expon.scaled = TRUE)), each = G), nrow = G))
}
mixture_grid <- function(theta, mu, kappa, weight) {
  stopifnot(nrow(mu) == 2L, identical(dim(mu), dim(kappa)),
            ncol(mu) == length(weight))
  W <- matrix(rep(weight, each = length(theta)), nrow = length(theta))
  W * vm_grid(theta, mu[1, ], kappa[1, ]) +
    (1 - W) * vm_grid(theta, mu[2, ], kappa[2, ])
}
mixture_at <- function(theta, state) {
  W <- state$weight
  stopifnot(is.matrix(W), identical(dim(W), dim(state$muone)))
  log_add(log(W) + vm_logpdf(theta, state$muone, state$kone),
          log1p(-W) + vm_logpdf(theta, state$mutwo, state$ktwo))
}
BLUE <- "#1F77B4"
ORANGE <- "#FF7F0E"
RED <- "#D62728"
HYP_COLORS <- c("Optic flow" = BLUE, "Vec. encoding" = ORANGE)
angle_breaks <- c(-pi, -pi / 2, 0, pi / 2, pi)
angle_labels <- c("-pi", "-pi/2", "0", "pi/2", "pi")


## Data loading & exploration

Read all `directional/*.parquet` files in sorted filename order from
[jgraving/unwrap-data](https://github.com/jgraving/unwrap-data). A supplied
`UNWRAP_DATA_ROOT` may identify a checkout or its `directional/` parts
directory. Otherwise sparsely clone that study into the repository's
ignored `unwrap-data/` directory.

The preprocessing below ports `unwrap/preprocess.py` literally: densely
interpolate at tolerance 0.0001 and repeatedly choose the remaining point
whose Euclidean distance from the current point is nearest six body lengths
($6\times0.04=0.24$). This is not cumulative arc-length resampling. Preserve
file/row order and keep the first nearest-distance tie, as in `np.argmin`.
Two of the 54 recordings yield no steps. Renumber the 52 survivors densely.


In [ ]:
data_override <- Sys.getenv("UNWRAP_DATA_ROOT", unset = "")
data_root <- if (nzchar(data_override)) data_override else file.path(repo_root, "unwrap-data")
find_parts <- function(root) {
  candidates <- c(file.path(root, "directional"), root)
  good <- vapply(candidates, function(p) {
    length(list.files(p, pattern = "[.]parquet$")) > 0L
  }, logical(1))
  if (any(good)) candidates[which(good)[1]] else NULL
}
data_dir <- find_parts(data_root)
if (is.null(data_dir)) {
  if (nzchar(data_override)) stop("UNWRAP_DATA_ROOT has no directional Parquet files.")
  if (!dir.exists(data_root)) {
    status <- system2("git", c("clone", "--depth", "1", "--filter=blob:none", "--sparse",
      "https://github.com/jgraving/unwrap-data.git", shQuote(data_root)))
    stopifnot(status == 0L)
  }
  status <- system2("git", c("-C", shQuote(data_root), "sparse-checkout", "add", "directional"))
  stopifnot(status == 0L)
  data_dir <- find_parts(data_root)
}
stopifnot(!is.null(data_dir))
files <- sort(list.files(data_dir, pattern = "[.]parquet$", full.names = TRUE), method = "radix")
parts <- lapply(files, function(file) {
  part <- as.data.frame(arrow::read_parquet(file, col_select = c("X", "Y", "fname")))
  part$fname <- as.character(part$fname)
  part$source_table <- if (startsWith(basename(file), "XY_raw_50cm_")) "XY_raw_50cm_" else "XY_raw_8cm_"
  part
})
raw <- do.call(rbind, parts)
rownames(raw) <- NULL
raw_ids <- unique(raw$fname)
raw$animal_id <- match(raw$fname, raw_ids) - 1L
cat("Raw files:", length(files), "recordings:", length(raw_ids), "rows:", nrow(raw), "\n")
stopifnot(length(files) == 54L, length(raw_ids) == 54L,
          all(is.finite(raw$X)), all(is.finite(raw$Y)))
# Python does not silently drop invalid XY rows; fail visibly if a new
# source version introduces them instead of changing the trajectories.
head(raw)


In [ ]:
rediscretize_trajectory <- function(x, y, step_length = 6 * 0.04, tolerance = 0.0001) {
  if (length(x) < 2L) return(data.frame(X = x, Y = y))
  counts <- pmax(as.integer(sqrt(diff(x)^2 + diff(y)^2) / tolerance), 2L)
  n_dense <- sum(counts) - length(counts) + 1L
  dense_x <- dense_y <- numeric(n_dense)
  cursor <- 1L
  for (i in seq_along(counts)) {
    sx <- seq(x[i], x[i + 1L], length.out = counts[i])
    sy <- seq(y[i], y[i + 1L], length.out = counts[i])
    if (i > 1L) {
      sx <- sx[-1L]
      sy <- sy[-1L]
    }
    take <- cursor + seq_along(sx) - 1L
    dense_x[take] <- sx
    dense_y[take] <- sy
    cursor <- cursor + length(sx)
  }
  new_x <- x[1]
  new_y <- y[1]
  while (length(dense_x) > 0L) {
    distances <- sqrt((dense_x - tail(new_x, 1))^2 + (dense_y - tail(new_y, 1))^2)
    j <- which.min(abs(distances - step_length))
    if (distances[j] < step_length - tolerance) break
    new_x <- c(new_x, dense_x[j])
    new_y <- c(new_y, dense_y[j])
    keep <- seq.int(j, length(dense_x))
    dense_x <- dense_x[keep]
    dense_y <- dense_y[keep]
    if (j == 1L) break
  }
  data.frame(X = new_x, Y = new_y)
}
rows_by_animal <- split(seq_len(nrow(raw)), factor(raw$fname, levels = raw_ids))
step_tables <- lapply(seq_along(raw_ids), function(i) {
  g <- raw[rows_by_animal[[i]], ]
  path <- rediscretize_trajectory(g$X, g$Y)
  if (nrow(path) < 2L) return(NULL)
  dx <- diff(path$X)
  dy <- diff(path$Y)
  data.frame(X = path$X[-1L], Y = path$Y[-1L], dX = dx, dY = dy,
    fname = raw_ids[i], direction = atan2(dy, dx),
    source_animal_id = i - 1L, file_path = g$source_table[1])
})
df <- do.call(rbind, Filter(Negate(is.null), step_tables))
survivors <- sort(unique(df$source_animal_id))
df$animal_id <- match(df$source_animal_id, survivors) - 1L
df$animal <- factor(df$animal_id, levels = seq_along(survivors) - 1L)
df$iteration <- ave(seq_len(nrow(df)), df$animal_id, FUN = seq_along)
rownames(df) <- NULL
num_individuals <- nlevels(df$animal)
summary_table <- data.frame(animal_id = seq_len(num_individuals) - 1L,
                            num_steps = as.integer(table(df$animal)))
cat("Surviving animals:", num_individuals, "heading steps:", nrow(df), "\n")
cat("Steps per animal:", range(summary_table$num_steps), "\n")
stopifnot(num_individuals == 52L, nrow(df) == 2435L,
          min(summary_table$num_steps) == 3L, max(summary_table$num_steps) == 204L,
          all(abs(sqrt(df$dX^2 + df$dY^2) - 0.24) <= 0.0001 + 1e-10))
# Reference counts are from the Python study text; they are checked here
# during local execution, not claimed to have been reproduced in this draft.
head(df)
head(summary_table)


In [ ]:
segments <- do.call(rbind, lapply(split(df, df$animal), function(g) {
  g <- g[order(g$iteration), ]
  n <- nrow(g)
  data.frame(animal_id = g$animal_id[-1L], iteration = g$iteration[-1L],
    x = g$X[-n], y = g$Y[-n], xend = g$X[-1L], yend = g$Y[-1L])
}))
tab20 <- c("#1F77B4", "#AEC7E8", "#FF7F0E", "#FFBB78", "#2CA02C", "#98DF8A",
           "#D62728", "#FF9896", "#9467BD", "#C5B0D5", "#8C564B", "#C49C94",
           "#E377C2", "#F7B6D2", "#7F7F7F", "#C7C7C7", "#BCBD22", "#DBDB8D",
           "#17BECF", "#9EDAE5")
trajectory_panel <- function(limit, points = FALSE) {
  p <- ggplot(segments, aes(x, y, colour = factor(animal_id %% 20L))) +
    geom_segment(aes(xend = xend, yend = yend), linewidth = 0.25) +
    scale_colour_manual(values = tab20) +
    coord_equal(xlim = c(-limit, limit), ylim = c(-limit, limit)) +
    labs(x = "X Coordinate (meters)", y = "Y Coordinate (meters)") +
    theme(legend.position = "none")
  if (points) p <- p + geom_point(data = df,
    aes(X, Y, colour = factor(animal_id %% 20L)), inherit.aes = FALSE, size = 0.6)
  p
}
p_trajectories <- trajectory_panel(30) + trajectory_panel(5, points = TRUE)
save_figure(p_trajectories, "locust_directional_trajectories", 12, 6)


## Model definition: hierarchical bimodal von Mises mixture

To model the potentially bimodal directional heading patterns of the marching locusts, we use a hierarchical bimodal von Mises mixture model. The model captures both population-level (shared) and individual-specific directional patterns, with link functions and priors keeping the parameters in their appropriate domains.

### Individual-level likelihood

For individual $i$, a heading $y \in (-\pi,\pi]$ has likelihood

$$
p(y \mid \Theta_i)
= \omega_i\,\mathcal M\!\bigl(y \mid \mu_{i,1},\kappa_{i,1}\bigr)
+ \bigl(1-\omega_i\bigr)\,\mathcal M\!\bigl(y \mid \mu_{i,2},\kappa_{i,2}\bigr),
$$

where

$$
\mathcal M(y \mid \mu,\kappa)
= \frac{1}{2\pi I_0(\kappa)}\exp\bigl(\kappa\cos(y-\mu)\bigr).
$$

Here:

- $\Theta_i = \{\mu_{i,1},\mu_{i,2},\kappa_{i,1},\kappa_{i,2},\omega_i\}$ are the parameters for individual $i$.
- $\mu_{i,k}$ are the component mean directions, $\mu_{i,k} \in (-\pi,\pi]$.
- $\kappa_{i,k} > 0$ are the component concentrations.
- $\omega_i \in (0,1)$ is the mixture weight for the first component (coded as `weights[i]` in the implementation; the second component has weight $1-\omega_i$).
- $I_0(\kappa)$ is the modified Bessel function of the first kind (order zero).

In code, the latent mixture is represented by `mu[i, k]` and `kappa[i, k]` for components $k \in \{1,2\}$ and `weights[i]` for $\omega_i$.

### Observations and dataset structure

- The full dataset is $Y = \{Y_1,\dots,Y_N\}$, where $N$ is the number of individuals.
- Each $Y_i = \{y_{[i,1]},\dots,y_{[i,M_i]}\}$ is the set of headings for individual $i$, with $M_i$ observations.
- Each $y_{[i,j]}$ is a single heading for individual $i$ at index $j$.

Assuming conditional independence given $\Theta_i$, the full-data likelihood factorizes as

$$
p(Y \mid \Theta)
= \prod_{i=1}^{N} \prod_{j=1}^{M_i} p\!\bigl(y_{[i,j]} \mid \Theta_i\bigr).
$$

This hierarchical setup allows the model to:

1. Capture **individual-specific patterns** through $\Theta_i$ and the data $Y_i$.
2. Share information across individuals via population-level parameters, improving estimates for poorly sampled animals while retaining structure for well-sampled ones.

### Hierarchical structure: population parameters and offsets

We separate population-level structure from individual-level deviations. Let

$$
\boldsymbol{\mu}_0 = (\mu_{0,1},\mu_{0,2}), \qquad
\boldsymbol{\kappa}_0 = (\kappa_{0,1},\kappa_{0,2}), \qquad
\omega_0 \in (0,1)
$$

denote the population-level mixture parameters, and

$$
\boldsymbol{\mu}_i = (\mu_{i,1},\mu_{i,2}), \qquad
\boldsymbol{\kappa}_i = (\kappa_{i,1},\kappa_{i,2}), \qquad
\omega_i
$$

the corresponding parameters for individual $i$.

Individual parameters are defined as population parameters plus unit-specific offsets on appropriate pre-link scales:

$$
\boldsymbol{\mu}_i = \boldsymbol{\mu}_0 + \mathbf{b}_i,
$$

$$
\operatorname{softplus}^{-1}\!\bigl(\boldsymbol{\kappa}_i\bigr)
= \operatorname{softplus}^{-1}\!\bigl(\boldsymbol{\kappa}_0\bigr) + \mathbf{a}_i,
$$

$$
\operatorname{logit}(\omega_i) = \operatorname{logit}(\omega_0) + c_i,
$$

where:

- $\mathbf{b}_i = (b_{[1,i]}, b_{[2,i]})$ are offsets for the component mean directions.
- $\mathbf{a}_i = (a_{[1,i]}, a_{[2,i]})$ are offsets for the component concentrations on the **pre–softplus** scale.
- $c_i$ is a scalar offset for the mixture weight on the **logit** scale.
- $\operatorname{softplus}(x) = \log(1+\exp x)$ and
  $\operatorname{logit}(\omega) = \log\!\bigl(\omega/(1-\omega)\bigr)$.

Componentwise, for $k \in \{1,2\}$,

$$
\mu_{i,k} = \mu_{0,k} + b_{[k,i]},
$$

$$
\operatorname{softplus}^{-1}(\kappa_{i,k})
= \operatorname{softplus}^{-1}(\kappa_{0,k}) + a_{[k,i]},
$$

and

$$
c_i = \operatorname{logit}(\omega_i) - \operatorname{logit}(\omega_0).
$$

In the implementation:

- `mu_pop_unsorted[k]` holds $\mu_{0,k}$ before sorting for identifiability.
- `mu[i, k]` corresponds to $\mu_{i,k}$.
- `kappa_log_pop[k]` corresponds to $\operatorname{softplus}^{-1}(\kappa_{0,k})$.
- `kappa_log[i, k]` corresponds to $\operatorname{softplus}^{-1}(\kappa_{i,k})$.
- `weights_logit_pop` is $\operatorname{logit}(\omega_0)$.
- `weights_logit[i]` is $\operatorname{logit}(\omega_i)$.

Setting all offsets to zero, $\mathbf{b}_i = \mathbf{0}$, $\mathbf{a}_i = \mathbf{0}$, $c_i = 0$, recovers the **population mixture**
$\Theta_0 = \{\boldsymbol{\mu}_0,\boldsymbol{\kappa}_0,\omega_0\}$.


### R implementation and prior correspondence

The likelihood is unchanged by a simultaneous permutation of its two
components. We fit the unsorted component vectors and then use the same
per-draw population-mean ordering for every component-specific quantity,
including **both weight columns**, matching Python cell 10. No ordering
constraint, altered prior, or second sorting of individual means is added.
The weight is `plogis(eta)` and its complement, not a two-logit softmax.

| Quantity | Bimodal prior | Unimodal prior |
| --- | --- | --- |
| Population angle(s) | VM(0, 0.000001), two independent | VM(0, 0.000001) |
| Angular pooling concentration | LogNormal(log(30), 0.1), per component | LogNormal(log(10), 0.1) |
| Animal angular deviation | VM(0, component pooling concentration) | VM(0, pooling concentration) |
| Population pre-softplus concentration | Normal(log(exp(2)-1), 0.5), per component | Normal(log(exp(1.5)-1), 0.5) |
| Concentration deviation scale | HalfNormal(1.5), per component | HalfNormal(1.5) |
| Standardized concentration deviations | Normal(0, 1), independent | Normal(0, 1), independent |
| Population weight logit | Normal(0, 0.5) | — |
| Weight-logit deviation scale | HalfNormal(3) | — |
| Standardized weight-logit deviations | Normal(0, 1), independent | — |

The uniform factors below bound circular coefficient vectors to one period;
they add constants only. Explicit vector log densities provide the exact von
Mises priors. Gaussian group terms are independent and non-centered. All
coefficient priors refer to uncentered intercepts. Inspect `make_stancode`
output for bounds, dimensions, target terms, and links before interpreting
local fits.


In [ ]:
locust_mix <- custom_family("locust_mix",
  dpars = c("mu", "mutwo", "kappa", "kappatwo", "omega"),
  links = c("identity", "identity", "softplus", "softplus", "logit"),
  lb = c(NA, NA, 0, 0, 0), ub = c(NA, NA, NA, NA, 1), type = "real")
locust_vm <- custom_family("locust_vm", dpars = c("mu", "kappa"),
  links = c("identity", "softplus"), lb = c(NA, 0), type = "real")
vm_functions <- stanvar(scode = "
  real locust_wrap(real x) {
    return atan2(sin(x), cos(x));
  }
  real locust_vm_lpdf(real y, real mu, real kappa) {
    return von_mises_lpdf(y | locust_wrap(mu), kappa);
  }
  real locust_vm_rng(real mu, real kappa) {
    return von_mises_rng(locust_wrap(mu), kappa);
  }
  real locust_vm_vect_lpdf(vector y, real mu, real kappa) {
    real lp = 0;
    for (i in 1:num_elements(y)) lp += locust_vm_lpdf(y[i] | mu, kappa);
    return lp;
  }
  real locust_mix_lpdf(real y, real mu, real mutwo,
                       real kappa, real kappatwo, real omega) {
    return log_mix(omega, locust_vm_lpdf(y | mu, kappa),
                         locust_vm_lpdf(y | mutwo, kappatwo));
  }
  real locust_mix_rng(real mu, real mutwo,
                     real kappa, real kappatwo, real omega) {
    if (bernoulli_rng(omega)) return locust_vm_rng(mu, kappa);
    return locust_vm_rng(mutwo, kappatwo);
  }
", block = "functions")
mix_hyper <- stanvar(scode = "real<lower=0> kmu_one; real<lower=0> kmu_two;",
                     block = "parameters")
# All animal coefficient vectors have the same level order, 0..51.
mix_form <- bf(direction ~ mu,
  nlf(mu ~ popone + devone),
  nlf(mutwo ~ poptwo + devtwo),
  popone ~ 1, poptwo ~ 1,
  devone ~ 0 + animal, devtwo ~ 0 + animal,
  kappa ~ 0 + Intercept + (1 | animal),
  kappatwo ~ 0 + Intercept + (1 | animal),
  omega ~ 0 + Intercept + (1 | animal),
  family = locust_mix)
mix_priors <- (
  prior("uniform(-pi(), pi())", class = "b", nlpar = "popone",
    lb = -3.141592653589793, ub = 3.141592653589793) +
  prior("uniform(-pi(), pi())", class = "b", nlpar = "poptwo",
    lb = -3.141592653589793, ub = 3.141592653589793) +
  prior("uniform(-pi(), pi())", class = "b", nlpar = "devone",
    lb = -3.141592653589793, ub = 3.141592653589793) +
  prior("uniform(-pi(), pi())", class = "b", nlpar = "devtwo",
    lb = -3.141592653589793, ub = 3.141592653589793) +
  set_prior("target += locust_vm_lpdf(b_popone[1] | 0, 1e-6)", check = FALSE) +
  set_prior("target += locust_vm_lpdf(b_poptwo[1] | 0, 1e-6)", check = FALSE) +
  set_prior("target += locust_vm_vect_lpdf(b_devone | 0, kmu_one)", check = FALSE) +
  set_prior("target += locust_vm_vect_lpdf(b_devtwo | 0, kmu_two)", check = FALSE) +
  set_prior("target += lognormal_lpdf(kmu_one | log(30), 0.1)", check = FALSE) +
  set_prior("target += lognormal_lpdf(kmu_two | log(30), 0.1)", check = FALSE) +
  prior("normal(log(exp(2) - 1), 0.5)", class = "b", coef = "Intercept", dpar = "kappa") +
  prior("normal(log(exp(2) - 1), 0.5)", class = "b", coef = "Intercept", dpar = "kappatwo") +
  prior("normal(0, 1.5)", class = "sd", dpar = "kappa") +
  prior("normal(0, 1.5)", class = "sd", dpar = "kappatwo") +
  prior("normal(0, 0.5)", class = "b", coef = "Intercept", dpar = "omega") +
  prior("normal(0, 3)", class = "sd", dpar = "omega")
)
mix_stanvars <- vm_functions + mix_hyper
mix_stan <- make_stancode(mix_form, data = df, prior = mix_priors, stanvars = mix_stanvars)
compact_mix_stan <- gsub("[[:space:]]", "", mix_stan)
stopifnot(grepl("target+=locust_mix_lpdf", compact_mix_stan, fixed = TRUE),
          grepl("lower=-3.141592653589793,upper=3.141592653589793", compact_mix_stan, fixed = TRUE),
          grepl("locust_vm_vect_lpdf(b_devone|0,kmu_one)", compact_mix_stan, fixed = TRUE),
          grepl("locust_vm_vect_lpdf(b_devtwo|0,kmu_two)", compact_mix_stan, fixed = TRUE))
for (name in c("popone", "poptwo", "devone", "devtwo")) {
  stopifnot(grepl(paste0("[K_", name, "]b_", name, ";"), compact_mix_stan, fixed = TRUE))
}
mix_design <- make_standata(mix_form, data = df, prior = mix_priors, stanvars = mix_stanvars)
stopifnot(mix_design$K_popone == 1L, mix_design$K_poptwo == 1L,
          mix_design$K_devone == num_individuals, mix_design$K_devtwo == num_individuals)
cat(mix_stan)


### Fit the bimodal model

Notebook execution is local by design (tests/test_notebooks.py).
NUTS on bounded circular vectors can start chains in separate wrapped
valleys. We initialize near two data-supported circular clusters, with
seeded per-chain jitter and each animal's deviations relative to those
population anchors. The anchors come from the observations, not the two
hypotheses. This changes the sampler's starting path only; every prior and
the posterior density remain unchanged. `seq_len(CHAINS)` creates one
initialization list for every chain.


In [ ]:
set.seed(SEED)
initial_clusters <- kmeans(cbind(cos(df$direction), sin(df$direction)),
                           centers = 2L, nstart = 20L)
cluster_means <- vapply(split(df$direction, initial_clusters$cluster), circ_mean, numeric(1))
cluster_order <- order(cluster_means)
anchor_mix <- cluster_means[cluster_order]
initial_group <- match(initial_clusters$cluster, cluster_order)
animal_offsets <- function(component) {
  vapply(levels(df$animal), function(id) {
    y <- df$direction[df$animal == id & initial_group == component]
    if (!length(y)) 0 else wrap(circ_mean(y) - anchor_mix[component])
  }, numeric(1))
}
anchor_offsets_one <- animal_offsets(1L)
anchor_offsets_two <- animal_offsets(2L)
mk_init_mix <- function(seed) {
  set.seed(seed)
  list(b_popone = as.numeric(wrap(anchor_mix[1] + runif(1, -0.15, 0.15))),
       b_poptwo = as.numeric(wrap(anchor_mix[2] + runif(1, -0.15, 0.15))),
       b_devone = as.numeric(wrap(anchor_offsets_one + runif(num_individuals, -0.1, 0.1))),
       b_devtwo = as.numeric(wrap(anchor_offsets_two + runif(num_individuals, -0.1, 0.1))),
       kmu_one = 30, kmu_two = 30,
       b_kappa = inverse_softplus(2), b_kappatwo = inverse_softplus(2), b_omega = 0)
}
init_mix <- lapply(seq_len(CHAINS), function(k) mk_init_mix(SEED + k))
stopifnot(length(init_mix) == CHAINS,
          all(vapply(init_mix, function(x) length(x$b_devone) == num_individuals &&
                       length(x$b_devtwo) == num_individuals, logical(1))))
set.seed(SEED)
fit_bimodal <- brm(mix_form, data = df, prior = mix_priors, stanvars = mix_stanvars,
  backend = "cmdstanr", chains = CHAINS, iter = ITER, warmup = WARMUP,
  cores = CHAINS, seed = SEED, init = init_mix,
  control = list(adapt_delta = 0.97, max_treedepth = 12))


In [ ]:
animal_grid <- data.frame(animal = factor(levels(df$animal), levels = levels(df$animal)))
extract_state <- function(fit, bimodal = TRUE) {
  d <- as.data.frame(posterior::as_draws_df(fit))
  get <- function(name) {
    stopifnot(name %in% names(d))
    as.numeric(d[[name]])
  }
  predict_matrix <- function(dpar, transform = FALSE) {
    x <- t(posterior_linpred(fit, newdata = animal_grid,
                            dpar = dpar, transform = transform))
    stopifnot(identical(dim(x), c(num_individuals, nrow(d))))
    x
  }
  m1 <- wrap(predict_matrix("mu"))
  k1 <- predict_matrix("kappa", transform = TRUE)
  p1 <- get("b_popone_Intercept")
  kp1 <- softplus(get("b_kappa_Intercept"))
  if (!bimodal) {
    return(list(draws = d, muone = m1, kone = k1,
      pop_muone = matrix(wrap(p1), nrow = 1L),
      pop_kone = matrix(kp1, nrow = 1L),
      quantities = c(list(mu_pop = wrap(p1), mu_kappa = get("kmu_one"),
        kappa_log_pop = get("b_kappa_Intercept"), kappa_pop = kp1,
        kappa_sigma = get("sd_animal__kappa_Intercept")),
        setNames(lapply(seq_len(num_individuals), function(i) m1[i, ]),
                 paste0("mu[", seq_len(num_individuals), "]")),
        setNames(lapply(seq_len(num_individuals), function(i) k1[i, ]),
                 paste0("kappa[", seq_len(num_individuals), "]")))))
  }
  m2 <- wrap(predict_matrix("mutwo"))
  k2 <- predict_matrix("kappatwo", transform = TRUE)
  w <- predict_matrix("omega", transform = TRUE)
  p2 <- get("b_poptwo_Intercept")
  kp2 <- softplus(get("b_kappatwo_Intercept"))
  wp <- plogis(get("b_omega_Intercept"))
  swap <- p1 > p2
  # Apply the population permutation ONCE per draw to every quantity.
  # Individual means are never sorted independently.
  reorder_pair <- function(a, b) {
    stopifnot(identical(dim(a), dim(b)), ncol(a) == length(swap))
    first <- a
    second <- b
    first[, swap] <- b[, swap, drop = FALSE]
    second[, swap] <- a[, swap, drop = FALSE]
    list(first, second)
  }
  means <- reorder_pair(m1, m2)
  kappas <- reorder_pair(k1, k2)
  pop_means <- reorder_pair(matrix(p1, nrow = 1L), matrix(p2, nrow = 1L))
  pop_kappas <- reorder_pair(matrix(kp1, nrow = 1L), matrix(kp2, nrow = 1L))
  pool_k <- reorder_pair(matrix(get("kmu_one"), nrow = 1L),
                         matrix(get("kmu_two"), nrow = 1L))
  sigma_k <- reorder_pair(matrix(get("sd_animal__kappa_Intercept"), nrow = 1L),
                          matrix(get("sd_animal__kappatwo_Intercept"), nrow = 1L))
  w[, swap] <- 1 - w[, swap, drop = FALSE]
  wp[swap] <- 1 - wp[swap]
  quantities <- list(logit_sigma = get("sd_animal__omega_Intercept"),
                     weights_pop = wp)
  for (k in 1:2) {
    quantities[[paste0("mu_pop[", k, "]")]] <- as.numeric(pop_means[[k]])
    quantities[[paste0("mu_kappa[", k, "]")]] <- as.numeric(pool_k[[k]])
    quantities[[paste0("kappa_pop[", k, "]")]] <- as.numeric(pop_kappas[[k]])
    quantities[[paste0("kappa_sigma[", k, "]")]] <- as.numeric(sigma_k[[k]])
    for (i in seq_len(num_individuals)) {
      quantities[[paste0("mu[", i, ",", k, "]")]] <- means[[k]][i, ]
      quantities[[paste0("kappa[", i, ",", k, "]")]] <- kappas[[k]][i, ]
    }
  }
  for (i in seq_len(num_individuals)) quantities[[paste0("weights[", i, "]")]] <- w[i, ]
  list(draws = d, muone = means[[1]], mutwo = means[[2]],
       kone = kappas[[1]], ktwo = kappas[[2]], weight = w,
       pop_muone = pop_means[[1]], pop_mutwo = pop_means[[2]],
       pop_kone = pop_kappas[[1]], pop_ktwo = pop_kappas[[2]],
       pop_weight = matrix(wp, nrow = 1L), quantities = quantities)
}
chain_matrix <- function(x, draws) {
  stopifnot(length(x) == nrow(draws))
  ids <- split(seq_len(nrow(draws)), draws[[".chain"]])
  stopifnot(length(ids) >= 2L, length(unique(lengths(ids))) == 1L)
  do.call(cbind, lapply(ids, function(ii) x[ii[order(draws[[".iteration"]][ii])]]))
}
diagnose_fit <- function(fit, state) {
  # Same retained-flag probe as the P1 helpers; absence is not zero.
  divergences <- tryCatch({
    samples <- fit$fit@sim$samples
    if (is.null(samples) || !length(samples)) {
      NA_real_
    } else {
      flags <- lapply(samples, function(ch) ch[["divergent"]])
      if (any(vapply(flags, is.null, logical(1)))) NA_real_ else sum(unlist(flags))
    }
  }, error = function(e) NA_real_)
  if (is.na(divergences)) {
    cat("Divergences: not retained in this fit object.\n")
  } else {
    cat("Divergences:", divergences, "\n")
    stopifnot(divergences == 0)
  }
  table <- do.call(rbind, lapply(names(state$quantities), function(p) {
    x <- chain_matrix(state$quantities[[p]], state$draws)
    stopifnot(all(is.finite(x)))
    angular <- grepl("^mu(\\[|_pop($|\\[))", p)
    z <- if (angular) wrap(x - circ_mean(x)) else x
    data.frame(parameter = p,
      mean = if (angular) circ_mean(x) else mean(x),
      sd = if (angular) sqrt(-2 * log(min(1, resultant(x)))) else sd(as.numeric(x)),
      rhat = if (angular) max(posterior::rhat(sin(x)), posterior::rhat(cos(x))) else posterior::rhat(x),
      ess_bulk = posterior::ess_bulk(z), row.names = NULL)
  }))
  # Angular R-hat uses the Rhat_unwrap sine/cosine convention. Every ESS
  # and R-hat calculation receives an [iteration x chain] matrix.
  print(table)
  stopifnot(all(is.finite(table$rhat)), all(table$rhat < 1.01),
            all(is.finite(table$ess_bulk)), all(table$ess_bulk > 400))
  table
}
trace_plot <- function(state, parameters) {
  long <- do.call(rbind, lapply(parameters, function(p) {
    x <- state$quantities[[p]]
    stopifnot(!is.null(x))
    if (grepl("^mu(\\[|_pop($|\\[))", p)) x <- wrap(x - circ_mean(x))
    data.frame(parameter = p, iteration = state$draws[[".iteration"]],
               chain = factor(state$draws[[".chain"]]), value = x)
  }))
  ggplot(long, aes(iteration, value, colour = chain, group = chain)) +
    geom_line(linewidth = 0.25, alpha = 0.65) +
    facet_wrap(~ parameter, scales = "free_y", ncol = 2) +
    labs(x = "retained iteration", y = "draw (angular traces centered)") +
    theme(legend.position = "bottom")
}
state_bimodal <- extract_state(fit_bimodal)
summary_bimodal <- diagnose_fit(fit_bimodal, state_bimodal)


In [ ]:
save_figure(trace_plot(state_bimodal,
  c("mu_pop[1]", "mu_pop[2]", "kappa_pop[1]", "kappa_pop[2]", "weights_pop", "logit_sigma")),
  "bimodal_trace", 9, 7)


## Hierarchical unimodal von Mises model

For comparison to the bimodal mixture, we also fit a hierarchical **unimodal** von Mises model in which each individual has a single preferred direction and concentration.

For individual $i$, a heading $y \in (-\pi,\pi]$ has likelihood
$$
p(y \mid \Theta_i)
= \mathcal M(y \mid \mu_i,\kappa_i),
$$
with
$$
\mathcal M(y \mid \mu,\kappa)
= \frac{1}{2\pi I_0(\kappa)} \exp\bigl(\kappa \cos(y-\mu)\bigr),
$$
where $\Theta_i = {\mu_i,\kappa_i}$, the mean direction $\mu_i \in (-\pi,\pi]$, the concentration $\kappa_i > 0$, and $I_0(\kappa)$ is the modified Bessel function of the first kind of order zero.

Let $Y_i = {y_{[i,1]},\dots,y_{[i,M_i]}}$ denote the headings for individual $i$, and $Y = {Y_1,\dots,Y_N}$ the full dataset from $N$ individuals.
As in the bimodal case, we separate population-level structure from individual-level deviations via a hierarchical parameterization.

This unimodal hierarchical model provides a simpler baseline in which each animal has a single preferred direction $\mu_i$ and concentration $\kappa_i$, regularized toward shared population values $(\mu_0,\kappa_0)$. It is used as a comparison point for the more flexible bimodal mixture model to assess whether the data warrant modeling two distinct directional modes.


In [ ]:
uni_hyper <- stanvar(scode = "real<lower=0> kmu_one;", block = "parameters")
uni_form <- bf(direction ~ mu,
  nlf(mu ~ popone + devone),
  popone ~ 1, devone ~ 0 + animal,
  kappa ~ 0 + Intercept + (1 | animal), family = locust_vm)
uni_priors <- (
  prior("uniform(-pi(), pi())", class = "b", nlpar = "popone",
    lb = -3.141592653589793, ub = 3.141592653589793) +
  prior("uniform(-pi(), pi())", class = "b", nlpar = "devone",
    lb = -3.141592653589793, ub = 3.141592653589793) +
  set_prior("target += locust_vm_lpdf(b_popone[1] | 0, 1e-6)", check = FALSE) +
  set_prior("target += locust_vm_vect_lpdf(b_devone | 0, kmu_one)", check = FALSE) +
  set_prior("target += lognormal_lpdf(kmu_one | log(10), 0.1)", check = FALSE) +
  prior("normal(log(exp(1.5) - 1), 0.5)", class = "b", coef = "Intercept", dpar = "kappa") +
  prior("normal(0, 1.5)", class = "sd", dpar = "kappa")
)
uni_stanvars <- vm_functions + uni_hyper
uni_stan <- make_stancode(uni_form, data = df, prior = uni_priors, stanvars = uni_stanvars)
compact_uni_stan <- gsub("[[:space:]]", "", uni_stan)
stopifnot(grepl("target+=locust_vm_lpdf", compact_uni_stan, fixed = TRUE),
          grepl("locust_vm_vect_lpdf(b_devone|0,kmu_one)", compact_uni_stan, fixed = TRUE),
          grepl("lognormal_lpdf(kmu_one|log(10),0.1)", compact_uni_stan, fixed = TRUE),
          grepl("lower=-3.141592653589793,upper=3.141592653589793", compact_uni_stan, fixed = TRUE))
for (name in c("popone", "devone")) {
  stopifnot(grepl(paste0("[K_", name, "]b_", name, ";"), compact_uni_stan, fixed = TRUE))
}
uni_design <- make_standata(uni_form, data = df, prior = uni_priors, stanvars = uni_stanvars)
stopifnot(uni_design$K_popone == 1L, uni_design$K_devone == num_individuals)
cat(uni_stan)


Notebook execution is local by design (tests/test_notebooks.py).
The unimodal fit uses the pooled circular mean and per-animal circular
means as initial anchors, with independent seeded jitter in every chain.
These starting values do not change its priors or likelihood.


In [ ]:
data_anchor <- circ_mean(df$direction)
individual_anchor <- vapply(split(df$direction, df$animal), circ_mean, numeric(1))
mk_init_uni <- function(seed) {
  set.seed(seed)
  list(b_popone = as.numeric(wrap(data_anchor + runif(1, -0.15, 0.15))),
       b_devone = as.numeric(wrap(individual_anchor - data_anchor +
                                 runif(num_individuals, -0.1, 0.1))),
       kmu_one = 10, b_kappa = inverse_softplus(1.5))
}
init_uni <- lapply(seq_len(CHAINS), function(k) mk_init_uni(SEED + k))
stopifnot(length(init_uni) == CHAINS)
set.seed(SEED)
fit_unimodal <- brm(uni_form, data = df, prior = uni_priors, stanvars = uni_stanvars,
  backend = "cmdstanr", chains = CHAINS, iter = ITER, warmup = WARMUP,
  cores = CHAINS, seed = SEED, init = init_uni,
  control = list(adapt_delta = 0.97, max_treedepth = 12))
state_unimodal <- extract_state(fit_unimodal, bimodal = FALSE)
summary_unimodal <- diagnose_fit(fit_unimodal, state_unimodal)


In [ ]:
save_figure(trace_plot(state_unimodal,
  c("mu_pop", "mu_kappa", "kappa_pop", "kappa_sigma")), "unimodal_trace", 9, 5)


## Model comparison: bimodal vs. unimodal

PSIS-LOO targets a held-out **step conditional on the observed animals**,
matching the Python pointwise log likelihood. It is not leave-one-animal-out
validation. Compute explicit log likelihoods in observation chunks, retaining
all posterior draws and chain IDs. Column scaling before exponentiation
prevents underflow in `loo::relative_eff` without changing relative
efficiency. Print Pareto-k diagnostics before interpreting the comparison.

Per-animal panels retain the Python helper's zero-length error bars: that
helper assigns zero SE to each individual pointwise contribution. Total
model ELPDs use the actual PSIS-LOO standard errors. The total paired
difference table additionally reports `sqrt(N * var(delta_pointwise))`.


In [ ]:
explicit_loo <- function(state, bimodal = TRUE, chunk_size = 128L) {
  S <- nrow(state$draws)
  ll <- matrix(NA_real_, nrow = S, ncol = nrow(df))
  for (start in seq.int(1L, nrow(df), by = chunk_size)) {
    ii <- seq.int(start, min(start + chunk_size - 1L, nrow(df)))
    a <- df$animal_id[ii] + 1L
    y <- matrix(rep(df$direction[ii], S), nrow = length(ii))
    l1 <- vm_logpdf(y, state$muone[a, , drop = FALSE], state$kone[a, , drop = FALSE])
    if (bimodal) {
      w <- state$weight[a, , drop = FALSE]
      l2 <- vm_logpdf(y, state$mutwo[a, , drop = FALSE], state$ktwo[a, , drop = FALSE])
      ll[, ii] <- t(log_add(log(w) + l1, log1p(-w) + l2))
    } else {
      ll[, ii] <- t(l1)
    }
  }
  stopifnot(all(is.finite(ll)))
  likelihood <- exp(sweep(ll, 2, apply(ll, 2, max), "-"))
  r_eff <- loo::relative_eff(likelihood, chain_id = state$draws[[".chain"]])
  result <- loo::loo(ll, r_eff = r_eff)
  print(result)
  print(loo::pareto_k_table(result))
  result
}
loo_bimodal <- explicit_loo(state_bimodal)
loo_unimodal <- explicit_loo(state_unimodal, bimodal = FALSE)
models <- list(Bimodal = loo_bimodal, Unimodal = loo_unimodal)
elpd_table <- do.call(rbind, lapply(names(models), function(name) {
  x <- models[[name]]
  data.frame(model = name, elpd = x$estimates["elpd_loo", "Estimate"],
    se = x$estimates["elpd_loo", "SE"], max_pareto_k = max(x$diagnostics$pareto_k),
    n_pareto_k_gt_0.7 = sum(x$diagnostics$pareto_k > 0.7))
}))
delta_pointwise <- loo_bimodal$pointwise[, "elpd_loo"] - loo_unimodal$pointwise[, "elpd_loo"]
print(elpd_table)
print(data.frame(comparison = "Bimodal - Unimodal", delta_elpd = sum(delta_pointwise),
                 se = sqrt(length(delta_pointwise) * var(delta_pointwise))))
p_elpd <- ggplot(elpd_table, aes(elpd, factor(model, levels = c("Unimodal", "Bimodal")),
                                 colour = model, shape = model)) +
  geom_errorbar(aes(xmin = elpd - se, xmax = elpd + se), orientation = "y", width = 0.15) +
  geom_point(size = 2.5) + scale_colour_manual(values = c(Bimodal = ORANGE, Unimodal = BLUE)) +
  scale_shape_manual(values = c(Bimodal = 15, Unimodal = 16)) +
  labs(x = "PSIS-LOO ELPD +/-1 SE", y = "") + theme(legend.position = "none")
save_figure(p_elpd, "locust_bimodal_unimodal_elpd", 6, 3)


In [ ]:
elpd_individual <- do.call(rbind, lapply(seq_len(num_individuals), function(i) {
  ii <- which(df$animal_id == i - 1L)
  data.frame(individual = i,
    bimodal = sum(loo_bimodal$pointwise[ii, "elpd_loo"]),
    unimodal = sum(loo_unimodal$pointwise[ii, "elpd_loo"]),
    delta = sum(delta_pointwise[ii]), se = 0)
}))
print(elpd_individual)
individual_levels <- rev(as.character(seq_len(num_individuals)))
elpd_long <- rbind(data.frame(individual = elpd_individual$individual,
                             value = elpd_individual$bimodal, model = "Bimodal"),
                   data.frame(individual = elpd_individual$individual,
                             value = elpd_individual$unimodal, model = "Unimodal"))
p_elpd_individual <- ggplot(elpd_long,
  aes(value, factor(individual, levels = individual_levels), colour = model, shape = model)) +
  geom_point(position = position_dodge(width = 0.5), size = 1.8) +
  scale_colour_manual(values = c(Bimodal = ORANGE, Unimodal = BLUE)) +
  scale_shape_manual(values = c(Bimodal = 15, Unimodal = 16)) +
  labs(x = "PSIS-LOO ELPD contribution", y = "Individual") + theme(legend.position = "bottom")
p_delta <- ggplot(elpd_individual,
  aes(delta, factor(individual, levels = individual_levels))) +
  geom_point(size = 1.8) + geom_vline(xintercept = 0, linetype = "dashed") +
  labs(x = "Delta ELPD (Bimodal - Unimodal)", y = "")
save_figure(p_elpd_individual + p_delta,
  "locust_elpd_by_individual_ab", 10, 12)


## Posterior predictive check: model density vs. non-parametric estimates

This posterior predictive check compares the **posterior predictive density** of the fitted hierarchical model with **non-parametric estimates** of the population-level directional density, using:

- **Posterior Predictive Density**: Using the posterior estimates from the hierarchical model to compute the overall directional density.
- **Bootstrapped Histogram Estimates**: Non-parametric bootstrap sampling to approximate the mean density for each directional bin.
- **Von Mises Kernel Density Estimates (KDE)**: Using a Von Mises kernel to estimate the density smoothly over the entire circle.

Together they show how well the fitted model reproduces the dominant population-level directional pattern.


Population density sets animal offsets to zero. The bootstrap is
hierarchical: sample animals with replacement, resample steps within each
selected animal, then average animal-specific estimates with equal weight.
Retain the Python constants: 16 bins, 360 grid points, population KDE
concentration 30, individual KDE concentration 15, and bootstrap seed 123.
Quick mode reduces repetitions only. R and NumPy random streams differ;
bootstrap realizations have distributional rather than bitwise parity.


In [ ]:
population_state <- list(muone = state_bimodal$pop_muone,
  mutwo = state_bimodal$pop_mutwo, kone = state_bimodal$pop_kone,
  ktwo = state_bimodal$pop_ktwo, weight = state_bimodal$pop_weight)
theta_grid <- seq(-pi, pi, length.out = 360L)
pop_mu <- rbind(as.numeric(population_state$muone), as.numeric(population_state$mutwo))
pop_kappa <- rbind(as.numeric(population_state$kone), as.numeric(population_state$ktwo))
pop_pdf <- mixture_grid(theta_grid, pop_mu, pop_kappa, as.numeric(population_state$weight))
stopifnot(identical(dim(pop_pdf), c(length(theta_grid), nrow(state_bimodal$draws))))
# Python population_density summarizes log density first, then exponentiates.
q_pop <- exp(t(apply(log(pop_pdf), 1, quantile, probs = c(0.025, 0.5, 0.975))))
model_density <- data.frame(theta = theta_grid,
  low = q_pop[, 1], center = q_pop[, 2], high = q_pop[, 3])
rm(pop_pdf)
rows_by_animal <- split(seq_len(nrow(df)), df$animal)
bins <- seq(-pi, pi, length.out = 17L)
bin_width <- diff(bins)[1]
bin_id <- findInterval(wrap(df$direction), bins, rightmost.closed = TRUE)
stopifnot(all(bin_id >= 1L & bin_id <= 16L))
hist_kernel <- matrix(0, nrow = 16L, ncol = nrow(df))
hist_kernel[cbind(bin_id, seq_len(nrow(df)))] <- 1 / bin_width
# Each column is one observed heading's VM kernel on the theta grid.
kde_kernel <- vm_grid(theta_grid, wrap(df$direction), rep(30, nrow(df)))
boot_hist <- matrix(0, nrow = 16L, ncol = N_BOOTSTRAPS)
boot_kde <- matrix(0, nrow = length(theta_grid), ncol = N_BOOTSTRAPS)
set.seed(123)
for (b in seq_len(N_BOOTSTRAPS)) {
  selected <- sample.int(num_individuals, num_individuals, replace = TRUE)
  weights <- numeric(nrow(df))
  for (i in selected) {
    ii <- rows_by_animal[[i]]
    draw <- sample.int(length(ii), length(ii), replace = TRUE)
    weights[ii] <- weights[ii] + tabulate(draw, nbins = length(ii)) /
      (num_individuals * length(ii))
  }
  stopifnot(abs(sum(weights) - 1) < 1e-10)
  boot_hist[, b] <- as.numeric(hist_kernel %*% weights)
  boot_kde[, b] <- as.numeric(kde_kernel %*% weights)
}
q_hist <- t(apply(boot_hist, 1, quantile, probs = c(0.025, 0.5, 0.975)))
q_kde <- t(apply(boot_kde, 1, quantile, probs = c(0.025, 0.5, 0.975)))
hist_density <- data.frame(theta = head(bins, -1L) + bin_width / 2,
  low = q_hist[, 1], center = q_hist[, 2], high = q_hist[, 3])
kde_density <- data.frame(theta = theta_grid,
  low = q_kde[, 1], center = q_kde[, 2], high = q_kde[, 3])
rm(hist_kernel, kde_kernel, boot_hist, boot_kde)
invisible(gc())
model_layer <- function() {
  ggplot(model_density, aes(theta, center)) +
    geom_ribbon(aes(ymin = low, ymax = high), fill = RED, alpha = 0.25) +
    geom_line(colour = RED, linewidth = 0.6) +
    scale_x_continuous(breaks = angle_breaks, labels = angle_labels,
                       limits = c(-pi, pi), expand = c(0, 0)) +
    labs(x = "Direction (radians)", y = "Density")
}
p_hist <- model_layer() +
  geom_errorbar(data = hist_density, aes(ymin = low, ymax = high), width = 0.06) +
  geom_point(data = hist_density, size = 1.5)
p_kde <- model_layer() +
  geom_ribbon(data = kde_density, aes(ymin = low, ymax = high), fill = "black", alpha = 0.15) +
  geom_line(data = kde_density, linetype = "dashed", linewidth = 0.5)
save_figure(p_hist / p_kde, "population_directional_validation_cartesian", 7, 7)


In [ ]:
edge_radius <- 0.5
p_validation_polar <- ggplot(model_density, aes(theta, edge_radius + center)) +
  geom_ribbon(aes(ymin = edge_radius + low, ymax = edge_radius + high), fill = RED, alpha = 0.25) +
  geom_line(colour = RED, linewidth = 0.6) +
  geom_ribbon(data = kde_density, aes(ymin = edge_radius + low, ymax = edge_radius + high),
              fill = "black", alpha = 0.15) +
  geom_line(data = kde_density, linetype = "dashed", linewidth = 0.5) +
  geom_errorbar(data = hist_density, aes(ymin = edge_radius + low, ymax = edge_radius + high), width = 0.02) +
  geom_point(data = hist_density, size = 1.3) +
  geom_hline(yintercept = edge_radius, linewidth = 0.3) +
  scale_x_continuous(limits = c(-pi, pi), breaks = angle_breaks,
                     labels = angle_labels, expand = c(0, 0)) +
  scale_y_continuous(limits = c(0, edge_radius + max(model_density$high, kde_density$high) * 1.2)) +
  coord_polar(start = -pi / 2, direction = -1) +
  labs(x = "Direction", y = "Density above radius 0.5") +
  theme(axis.text.y = element_blank(), axis.ticks.y = element_blank())
save_figure(p_validation_polar, "population_directional_validation", 7, 7)


## Individual-level posterior predictive checks

The individual-level posterior predictive check compares each animal's observed headings with the posterior predictive density of the fitted model. Arrows mark the two mixture-component mean directions, scaled by each component's contribution to the expected resultant vector (its mixture weight times its mean resultant length); inward residuals scale the pointwise lack of fit against the local predictive density.


The component arrows and plug-in mean PDF reproduce the Python helper's
`wrap(mu_flat.mean(axis=0))` (arithmetic angle mean followed by wrapping),
rather than silently replacing it with a circular mean. The 95% band is
computed from complete posterior density draws. Residual ticks use the
inverse plug-in density normalized by its trapezoidal integral, and point
outward for observations and inward for residuals. Animal labels are 1–52
in the figures, corresponding to dense data IDs 0–51.


In [ ]:
individual_theta <- -pi + (0:359) * (2 * pi / 360)
individual_curves <- vector("list", num_individuals)
individual_ticks <- vector("list", num_individuals)
individual_arrows <- vector("list", num_individuals)
for (i in seq_len(num_individuals)) {
  y <- wrap(df$direction[rows_by_animal[[i]]])
  mu <- rbind(state_bimodal$muone[i, ], state_bimodal$mutwo[i, ])
  kappa <- rbind(state_bimodal$kone[i, ], state_bimodal$ktwo[i, ])
  weight <- state_bimodal$weight[i, ]
  pdf <- mixture_grid(individual_theta, mu, kappa, weight)
  bounds <- t(apply(pdf, 1, quantile, probs = c(0.025, 0.975)))
  # Source semantics: PDF of posterior mean parameters, not mean PDF.
  mu_mean <- wrap(rowMeans(mu))
  kappa_mean <- rowMeans(kappa)
  w_mean <- mean(weight)
  plugin <- as.numeric(mixture_grid(individual_theta,
    matrix(mu_mean, ncol = 1L), matrix(kappa_mean, ncol = 1L), w_mean))
  kernel <- vm_grid(individual_theta, y, rep(15, length(y)))
  boot <- matrix(0, nrow = length(individual_theta), ncol = N_BOOT_KDE)
  set.seed(123L + 10000L + i - 1L)
  for (b in seq_len(N_BOOT_KDE)) {
    boot[, b] <- rowMeans(kernel[, sample.int(length(y), length(y), replace = TRUE), drop = FALSE])
  }
  kde <- t(apply(boot, 1, quantile, probs = c(0.025, 0.5, 0.975)))
  curve <- data.frame(individual = i, theta = individual_theta,
    model = plugin, model_low = bounds[, 1], model_high = bounds[, 2],
    kde_low = kde[, 1], kde = kde[, 2], kde_high = kde[, 3])
  # Close the periodic curve explicitly with the first grid value.
  end <- curve[1, ]
  end$theta <- pi
  individual_curves[[i]] <- rbind(curve, end)
  inv_grid <- 1 / plugin
  integral <- sum(diff(individual_theta) *
                  (head(inv_grid, -1L) + tail(inv_grid, -1L)) / 2)
  at_data <- as.numeric(mixture_grid(y, matrix(mu_mean, ncol = 1L),
                                     matrix(kappa_mean, ncol = 1L), w_mean))
  individual_ticks[[i]] <- data.frame(individual = i, theta = y,
    residual = -1 / (at_data * (integral + 1e-12)))
  individual_arrows[[i]] <- data.frame(individual = i, theta = mu_mean,
    length = c(w_mean, 1 - w_mean) * A1_safe(kappa_mean))
}
ppc_curves <- do.call(rbind, individual_curves)
ppc_ticks <- do.call(rbind, individual_ticks)
ppc_arrows <- do.call(rbind, individual_arrows)
ppc_levels <- seq_len(num_individuals)
for (name in c("ppc_curves", "ppc_ticks", "ppc_arrows")) {
  object <- get(name)
  object$individual <- factor(object$individual, levels = ppc_levels,
                              labels = paste("Individual", ppc_levels))
  assign(name, object)
}
rmax <- 1 + max(ppc_curves$model_high, ppc_curves$kde_high) * 1.15
p_individual_ppc <- ggplot(ppc_curves, aes(theta, 1 + model, group = individual)) +
  geom_hline(yintercept = c(1, 2), colour = "grey40", linewidth = 0.25) +
  geom_vline(xintercept = c(-pi, -pi / 2, 0, pi / 2), colour = "grey70", linetype = "dashed", linewidth = 0.2) +
  geom_ribbon(aes(ymin = 1 + kde_low, ymax = 1 + kde_high), fill = "grey50", alpha = 0.25) +
  geom_line(aes(y = 1 + kde), colour = "grey20", linetype = "dashed", linewidth = 0.4) +
  geom_ribbon(aes(ymin = 1 + model_low, ymax = 1 + model_high), fill = RED, alpha = 0.3) +
  geom_line(colour = RED, linewidth = 0.5) +
  geom_segment(data = ppc_ticks,
    aes(x = theta, xend = theta, y = 1, yend = 1.1), inherit.aes = FALSE, linewidth = 0.2) +
  geom_segment(data = ppc_ticks,
    aes(x = theta, xend = theta, y = 1, yend = 1 + residual),
    inherit.aes = FALSE, colour = "#17BECF", linewidth = 0.2) +
  geom_segment(data = ppc_arrows,
    aes(x = theta, xend = theta, y = 0, yend = length), inherit.aes = FALSE,
    colour = RED, linewidth = 0.4, arrow = grid::arrow(length = grid::unit(0.05, "inches"))) +
  scale_x_continuous(limits = c(-pi, pi), expand = c(0, 0)) +
  scale_y_continuous(limits = c(0, rmax), expand = c(0, 0)) +
  coord_polar(start = -pi / 2, direction = -1) +
  facet_wrap(~ individual, ncol = 7) + theme_void(base_size = 8) +
  theme(strip.text = element_text(size = 8), panel.spacing = grid::unit(0.5, "lines"))
save_figure(p_individual_ppc, "locust_directional_individual_ppc", 14, 16)


## Bayesian hypothesis tests

This section quantifies how much **posterior predictive density** falls near the theory-predicted directions, relative to a uniform baseline:

1. **Optic flow (forward) hypothesis** ($y = 0$): the locusts align with the wide-field motion and keep marching forward.
2. **Vectorial encoding (lateral) hypothesis** ($y = \pm \frac{\pi}{2}$): the locusts commit to one of the two lateral virtual swarms.
3. **Random walk baseline**: uniform density $p_{\text{unif}} = 1/(2\pi)$ - no directional bias.

**Model definition for the hypothesis tests.** Using the posterior samples, we construct the **population-level mixture model** by setting the individual offsets to zero, so the model is described by the **population-level parameter vector** $ \Theta_0 = \{\omega_0, \boldsymbol{\kappa}_0, \boldsymbol{\mu}_0\} $, where:
- $ \omega_0 $ (coded as `weights_pop`) is the mixture weight for the two directional modes,
- $ \boldsymbol{\kappa}_0 = \{\kappa_{0,1}, \kappa_{0,2}\} $ (coded as `kappa_pop`) are the component concentrations, and
- $ \boldsymbol{\mu}_0 = \{\mu_{0,1}, \mu_{0,2}\} $ (coded as `mu_pop`) are the component mean directions.

The **posterior predictive density** is the normalized componentwise mixture

$$
p(y \mid \Theta_0)
= \omega_0\,\mathcal M(y\mid\mu_{0,1},\kappa_{0,1})
+ (1-\omega_0)\,\mathcal M(y\mid\mu_{0,2},\kappa_{0,2}).
$$

**Formula correction:** the shared-$Z$ display in Python cell 25 is not a
normalizer for components with different concentrations. This expression
matches the actual Python likelihood, its post-hoc computations, and the
correct expression in Python cell 26. No computation or estimand is changed.

**Hypothesis values.**

- **Optic flow (forward)**: $p_{\to} = p(y = 0 \mid \Theta_0)$.
- **Vectorial encoding (lateral)**: the per-mode average $p_{\updownarrow} = \frac{1}{2} \left[ p(y = \frac{\pi}{2} \mid \Theta_0) + p(y = -\frac{\pi}{2} \mid \Theta_0) \right]$; the total density assigned to the lateral hypothesis is twice the reported per-mode value, which keeps it directly comparable to the single-mode optic-flow value.
- Comparing both against the random walk baseline shows whether the fitted model supports a directional preference for forward or lateral movement.

The same evaluation is repeated per animal, using each animal's posterior parameter vector instead of $\Theta_0$.


The following figures evaluate density at the hypotheses and compare
weight-only alignment scores. These are distinct from the held-out-step
PSIS-LOO scores above. The plots labeled “ELPD” in the Python twin use the
mean **per-draw log density at a fixed hypothesis direction**, with bars
`1.96 * sd(log_density) / sqrt(number_of_draws)`. We preserve that arithmetic;
these bars are not effective-sample-size-adjusted posterior intervals.

For the lateral hypothesis, combine densities with a **per-mode average**:
`log_add(log_p_left, log_p_right) - log(2)`. The pooled individual-density row
is an average of individual **log** densities within each draw, followed by
summarization and exponentiation, matching the source; it is not an
arithmetic mixture of individual densities. The population sets offsets to
zero, and is therefore distinct from this pooled row.


In [ ]:
hypothesis_arrays <- function(state) {
  forward <- mixture_at(0, state)
  left <- mixture_at(-pi / 2, state)
  right <- mixture_at(pi / 2, state)
  list(forward = forward, perpendicular = log_add(left, right) - log(2))
}
hyp_pop <- hypothesis_arrays(population_state)
hyp_ind <- hypothesis_arrays(state_bimodal)
interval_row <- function(x, mode = c("density", "elpd", "alignment")) {
  mode <- match.arg(mode)
  if (mode == "elpd") {
    center <- mean(x)
    error <- 1.96 * sd(x) / sqrt(length(x))
    return(c(center = center, low = center - error, high = center + error))
  }
  bounds <- hdi(x)
  if (mode == "density") {
    return(c(center = exp(median(x)), low = exp(unname(bounds[1])), high = exp(unname(bounds[2]))))
  }
  c(center = mean(x), low = unname(bounds[1]), high = unname(bounds[2]))
}
make_hyp_table <- function(arrays, mode, pooled = FALSE) {
  f <- arrays$forward
  p <- arrays$perpendicular
  stopifnot(is.matrix(f), identical(dim(f), dim(p)))
  labels <- as.character(seq_len(nrow(f)))
  if (pooled) {
    f <- rbind(f, colMeans(f))
    p <- rbind(p, colMeans(p))
    labels <- c(labels, "Pooled")
  }
  do.call(rbind, lapply(seq_len(nrow(f)), function(i) {
    z <- rbind(interval_row(f[i, ], mode), interval_row(p[i, ], mode))
    data.frame(individual = labels[i], hypothesis = c("Optic flow", "Vec. encoding"),
      center = z[, 1], low = z[, 2], high = z[, 3],
      probability = mean(p[i, ] > f[i, ]), row.names = NULL)
  }))
}
population_dots <- function(table, label, baseline) {
  ggplot(table, aes(center, factor(hypothesis, levels = c("Vec. encoding", "Optic flow")),
                    colour = hypothesis, shape = hypothesis)) +
    geom_errorbar(aes(xmin = low, xmax = high), orientation = "y", width = 0.15) +
    geom_point(size = 2.5) + geom_vline(xintercept = baseline, linetype = "dashed") +
    scale_colour_manual(values = HYP_COLORS) +
    scale_shape_manual(values = c("Optic flow" = 16, "Vec. encoding" = 15)) +
    labs(x = label, y = "") + theme(legend.position = "none")
}
individual_dots <- function(table, label, baseline, limits = NULL) {
  labels <- unique(table$individual)
  table$individual <- factor(table$individual, levels = rev(labels))
  dodge <- position_dodge(width = 0.5)
  main <- ggplot(table, aes(center, individual, colour = hypothesis, shape = hypothesis)) +
    geom_errorbar(aes(xmin = low, xmax = high), orientation = "y", width = 0.2, position = dodge) +
    geom_point(size = 1.6, position = dodge) +
    geom_vline(xintercept = baseline, linetype = "dashed") +
    scale_colour_manual(values = HYP_COLORS) +
    scale_shape_manual(values = c("Optic flow" = 16, "Vec. encoding" = 15)) +
    labs(x = label, y = "Individual", tag = "a.") + theme(legend.position = "bottom")
  if (!is.null(limits)) main <- main + coord_cartesian(xlim = limits)
  probs <- table[!duplicated(table$individual), ]
  probability <- ggplot(probs, aes(probability, individual)) +
    geom_point(size = 1.6) + geom_vline(xintercept = 0.5, linetype = "dashed") +
    geom_vline(xintercept = 1) + scale_x_continuous(limits = c(0, 1.02)) +
    labs(x = "P(Vec. encoding > Optic flow)", y = "", tag = "b.") +
    theme(axis.text.y = element_blank(), axis.ticks.y = element_blank())
  main + probability + plot_layout(widths = c(1.35, 1))
}
pop_density_table <- make_hyp_table(hyp_pop, "density")
print(pop_density_table)
save_figure(population_dots(pop_density_table, "Predictive density", 1 / (2 * pi)),
  "locust_directional_hypothesis_horizontal", 6, 3)


In [ ]:
pop_elpd_table <- make_hyp_table(hyp_pop, "elpd")
print(pop_elpd_table)
save_figure(population_dots(pop_elpd_table, "Mean log predictive density at hypothesis", -log(2 * pi)),
  "locust_directional_hypothesis_horizontal_elpd", 6, 3)


In [ ]:
individual_density_table <- make_hyp_table(hyp_ind, "density", pooled = TRUE)
print(individual_density_table)
save_figure(individual_dots(individual_density_table, "Predictive density", 1 / (2 * pi)),
  "locust_directional_hypothesis_individual_ab", 10, 13)


In [ ]:
individual_elpd_table <- make_hyp_table(hyp_ind, "elpd", pooled = TRUE)
print(individual_elpd_table)
save_figure(individual_dots(individual_elpd_table, "Mean log predictive density at hypothesis", -log(2 * pi)),
  "locust_directional_hypothesis_individual_elpd_ab", 10, 13)


In [ ]:
alignment_arrays <- function(state) {
  w <- state$weight
  stopifnot(is.matrix(w), identical(dim(w), dim(state$muone)))
  list(forward = w * cos(state$muone) + (1 - w) * cos(state$mutwo),
       perpendicular = w * cos(2 * (state$muone - pi / 2)) +
         (1 - w) * cos(2 * (state$mutwo - pi / 2)))
}
# Weight-only alignment is the source estimand; no A1(kappa) factor.
align_pop <- alignment_arrays(population_state)
align_ind <- alignment_arrays(state_bimodal)
pop_alignment_table <- make_hyp_table(align_pop, "alignment")
print(pop_alignment_table)
save_figure(population_dots(pop_alignment_table, "Posterior mean alignment", 0) +
              coord_cartesian(xlim = c(-0.05, 1)),
  "locust_directional_alignment_horizontal", 6, 3)


In [ ]:
individual_alignment_table <- make_hyp_table(align_ind, "alignment")
print(individual_alignment_table)
save_figure(individual_dots(individual_alignment_table, "Posterior mean alignment", 0,
                            limits = c(-1, 1)),
  "locust_directional_alignment_individual_ab", 10, 13)


## Summary

This R port preserves the hierarchical two-component von Mises analysis,
the controlled unimodal baseline, conditional step-level PSIS-LOO, and all
population and individual predictive/hypothesis figures from the Python
example. Interpret the empirical support for optic flow versus vectorial
encoding only after the local divergence reports, hard R-hat/ESS gates,
Pareto-k diagnostics, and predictive plots have been reviewed.

Animal offsets are zero for population-level tests; individual checks retain
them. Predictive density, mean log density at a fixed theory-predicted
heading, and weight-only directional alignment answer different questions.
All results in this draft await local execution.
